# ITSN1 Gene-Level Meta-Analysis

- Last Updated: 7/25/2026
- Description: Runs meta-analysis using REMETA for burden testing of missense and LoF variants in the gene ITSN1. Burden tests run are remeta burden and SKAT/SKAT-O
- Data: GP2 R11 WGS and AoU WGS R8

## Imports

In [6]:
# System
import os
import sys

# Dataframe and array manipulation
import pandas as pd
import numpy as np
import math

# Path management
import pathlib
from pathlib import Path
import glob

# Statistics
import statsmodels.api as sm
import scipy
from scipy import stats

# Process management
import shutil
import subprocess

# Workflow management

# Visualization
#import matplotlib
#import matplotlib.pyplot as plt
#import seaborn as sns

# Date and time
from datetime import datetime

# Get the timestamp
d = datetime.now().strftime("%m%d%Y")
t = datetime.now().strftime("%m%d%Y_%H%M%S")
print("Last iteration date:", d, "EST")
print("Last iteration time:", t, "EST")

# Get package versions
packages = [pd, np, sm, scipy]

print("\nPackage versions:\n")
for p in packages:
    print("-", p.__name__, p.__version__)

Last iteration date: 07232026 EST
Last iteration time: 07232026_175137 EST

Package versions:

- pandas 2.3.2
- numpy 2.2.6
- statsmodels.api 0.14.6
- scipy 1.15.2


## Environment Setup

In [ ]:
BASE_DIR = f"/home/jupyter/ITSN1_LD_Matrices"

ancestry_list = ["AAC", "AFR", "AJ", "AMR", "EUR"]
GENE = "ITSN1"
HOME = "/home/jupyter"
DATA_DIR = f"{HOME}/workspace/"
RELEASE = str(8)
STARTBP = "33642501"
ENDBP = "33899861"
CHROM= str(21)
DT = "wgs"
TOOLS_DIR = f"/opt/workbench-tools/binaries/bin"
for ancestry in ancestry_list:
    os.makedirs(f"{BASE_DIR}/{ancestry}", exist_ok=True)

## Meta-Analysis with Regenie and Remeta
- Code shows generation of AoU remeta input files, but GP2 files were generated with the same process

### Create Annotation Files

In [12]:
# create annotation file for remeta analysis

# read in variant files

lof = pd.read_csv(f"{BASE_DIR}/AOU_WGS_LoF_variants_vep_for_remeta.csv")
missense = pd.read_csv(f"{BASE_DIR}/AOU_WGS_missense_variants_vep_for_remeta.csv")
mod_impact = pd.read_csv(f"{BASE_DIR}/AOU_WGS_mod_high_impact_variants_vep_for_remeta.csv")

In [13]:
# create annotation file
df_list = []
def make_remeta_anno(df, dt=""):
    # add datatype as third column
    anno_file = pd.DataFrame(df["#Uploaded_variation"].str.lstrip(to_strip="chr"), columns=None)
    anno_file["Gene"] = GENE
    anno_file["Type"] = dt
    return anno_file

In [14]:
lof_anno = make_remeta_anno(lof, "LoF")
missense_anno = make_remeta_anno(missense, "missense")


In [16]:
final_anno = pd.concat([lof_anno, missense_anno], axis=0)
final_anno.drop_duplicates(inplace=True)

In [17]:
# write final anno to file
final_anno.sort_values(by="#Uploaded_variation").to_csv(f"{BASE_DIR}/AOU_remeta_anno_file.txt", header=False, sep="\t", index=False)

### Format Pfiles
- Pfiles should have IDs as "chr:pos:ref:alt" ex: "21:00000:A:G" NOT "chr21:00000:A:G"

In [ ]:
# use annotation file to extract variants from each ancestry pfile set

for ancestry in ancestry_list:
    PLINK_PREFIX = f"chr{CHROM}_{ancestry}"
    WORK_DIR = f"{BASE_DIR}/{ancestry}"
    ! {TOOLS_DIR}/plink2 \
    --pfile /home/jupyter/{ancestry}_R8_wgs/{PLINK_PREFIX} \
    --set-all-var-ids '@:#:$r:$a' \ 
    --output-chr 26 \
    --new-id-max-allele-len 999 \
    --mac 1 \
    --make-pgen psam-cols=fid,maybesid,maybeparents,sex,phenos \
    --out {WORK_DIR}/chr21_{ancestry}

### Create LD Matrices and REMETA input files

In [26]:
# make a gene list
gene_list = [GENE, CHROM, STARTBP, ENDBP]
gene_str = "\t".join(gene_list)
with open(f"{BASE_DIR}/remeta_gene_list.txt", "w") as f:
        f.write(f"{gene_str}")


In [4]:
# make variant extract lists
for ancestry in ancestry_list:
    WORK_DIR=f"{BASE_DIR}/{ancestry}"
    
    # read in all annotated variants from results
    results = pd.read_csv(f"{BASE_DIR}/AOU_WGS.csv", usecols=["SNP", "Ancestry"])
    
    # subset by ancestry
    ancestry_results = results[results["Ancestry"] == ancestry]
    
    # write snps to extract file per ancestry
    variants = ancestry_results["SNP"].str.strip("chr")
    variants.sort_values(inplace=True)
    variants.to_csv(f"{WORK_DIR}/{ancestry}_extract.txt", index=False, header=False)

In [21]:
# write mask file
pd.DataFrame.from_dict({"LoF":"LoF",
              "missense":"missense",
              "LoF_missense":"LoF,missense"}, orient="index").to_csv(f"{BASE_DIR}/mask_file.txt", index=True, header=False, sep="\t")

### Make LD matrices
- no buffer option

In [ ]:
%%bash
# run shell script in terminal after activating remeta environment
for ancestry in AAC AFR AJ AMR EUR
do 
    WORK_DIR=/home/jupyter/ITSN1_LD_Matrices/${ancestry}
    ~/tools/remeta/build/remeta compute-ref-ld \
    --target-pfile ${WORK_DIR}/chr21_${ancestry} \
    --target-extract ${WORK_DIR}/${ancestry}_extract.txt \
    --gene-list /home/jupyter/ITSN1_LD_Matrices/remeta_gene_list.txt \
    --chr 21 \
    --out ${WORK_DIR}/${ancestry}_AOU.cohort \
    --threads 2
done

### Make htp files with REGENIE

In [ ]:
# Make regenie htp file
for ancestry in ["AAC"]:
    WORK_DIR=f"/home/jupyter/ITSN1_LD_Matrices/{ancestry}"
    ! /opt/workbench-tools/binaries/bin/regenie \
    --step 1 \
    --pgen {WORK_DIR}/{ancestry}_regenie \
    --covarFile {WORK_DIR}/{ancestry}_covariate_regenie.txt \
    --covarColList SEX,PC1,PC2,PC3,PC4,PC5 \
    --phenoFile {WORK_DIR}/{ancestry}_pheno_regenie.txt \
    --phenoCol PHENO \
    --bsize 100 \
    --firth \
    --minMac 1 \
    --cc12 \ 
    --bt --lowmem \
    --lowmem-prefix tmp_rg \
    --out {ancestry}_fit_bin_out

In [ ]:
# generate set-lists from new htp files
for ancestry in ["AAC", "AFR", "AJ", "AMR", "EUR"]:
    WORK_DIR=f"{BASE_DIR}/{ancestry}"
    htp = pd.read_csv(f"{WORK_DIR}/{ancestry}_step2_logi_PHENO.regenie", sep="\t")
    # sort by ascending position
    htp = htp.sort_values(by=["Pos"], ignore_index=True)
    htp["Gene"] = "ITSN1"
    set_list = ["ITSN1", "21", "33642501", ",".join(list(htp["Name"]))]
    with open(f"{WORK_DIR}/{ancestry}_set_list_regenie.txt", "w") as f:
        f.write("\t".join(set_list))
    

### create htp files with regenie

In [3]:
# generate set-lists
for ancestry in ["AAC", "AFR", "AJ", "AMR", "EUR"]:
    WORK_DIR=f"{BASE_DIR}/{ancestry}"
    htp = pd.read_csv(f"{WORK_DIR}/{ancestry}_htp.txt", sep="\t")
    # sort by ascending position
    htp = htp.sort_values(by=["Pos"], ignore_index=True)
    htp["Gene"] = "ITSN1"
    set_list = ["ITSN1","21", "33642501", ",".join(list(htp["Name"]))]
    with open(f"{WORK_DIR}/{ancestry}_set_list.txt", "w") as f:
        f.write("\t".join(set_list))
    
  

In [ ]:
# run regenie step 2

for ancestry in ancestry_list:
    WORK_DIR = f"{BASE_DIR}/{ancestry}"
    ! {TOOLS_DIR}/regenie \
    --step 2 \
    --pgen {WORK_DIR}/chr21_{ancestry} \
    --phenoFile {WORK_DIR}/{ancestry}_pheno_regenie.txt \
    --covarFile {WORK_DIR}/{ancestry}_covariate_regenie.txt \
    --covarColList SEX,PC1,PC2,PC3,PC4,PC5 \
    --set-list {WORK_DIR}/{ancestry}_set_list.txt \
    --extract {WORK_DIR}/{ancestry}_extract.txt \
    --ignore-pred \
    --write-mask-snplist \
    --check-burden-files \
    --bsize 200 \
    --af-cc \
    --cc12 \
    --bt \
    --minMAC 1 \
    --firth --firth-se --approx --pThresh 0.01 \
    --htp {COHORT} \
    --out {WORK_DIR}/{ancestry}_step2_logi

## Meta-analysis Merging GP2 and AoU

### Merge GP2 and AoU for Meta Analysis

In [ ]:
# merge HTP files and generate merged set lists
for ancestry in ["AAC", "AFR", "AJ", "AMR", "EUR"]:
    GP2_DIR = "/home/jupyter/ITSN1_LD_Matrices/GP2_remeta_final"
    AOU_DIR = "/home/jupyter/ITSN1_LD_Matrices"
    MERGE_DIR = "/home/jupyter/ITSN1_LD_Matrices/aou_gp2_merge_remeta_files"
    # htp files
    gp2_htp = pd.read_csv(f"{GP2_DIR}/{ancestry}_step2_logi_PHENO.regenie", sep="\t")
    aou_htp = pd.read_csv(f"{AOU_DIR}/{ancestry}/{ancestry}_step2_logi_PHENO.regenie", sep="\t")

    # sort both files and write to file
    gp2_htp = gp2_htp.sort_values(by=["Pos"], ignore_index=True)
    aou_htp = aou_htp.sort_values(by=["Pos"], ignore_index=True)
    gp2_htp.to_csv(f"{MERGE_DIR}/{ancestry}_htp_GP2.txt", sep="\t", index=False)
    aou_htp.to_csv(f"{MERGE_DIR}/{ancestry}_htp_AOU.txt", sep="\t", index=False)
    
    gp2_aou_merge_htp = pd.concat([gp2_htp, aou_htp], axis=0, ignore_index=True)
    gp2_aou_merge_htp = gp2_aou_merge_htp.sort_values(by=["Pos"], ignore_index=True)
    gp2_aou_merge_htp.to_csv(f"{MERGE_DIR}/{ancestry}_htp.txt",
                             sep="\t", index=False)

    # generate merged setlists
    gp2_aou_merge_htp["Gene"] = "ITSN1"
    set_list = ["ITSN1","21", "33642501", ",".join(list(gp2_aou_merge_htp["Name"]))]
    with open(f"{MERGE_DIR}/{ancestry}_set_list.txt", "w") as f:
        f.write("\t".join(set_list))

    # merge extract files
    ! cat {GP2_DIR}/{ancestry}_extract.txt {AOU_DIR}/{ancestry}/{ancestry}_extract.txt > {MERGE_DIR}/{ancestry}_extract.txt

    # merge anno files
    ! cat {GP2_DIR}/GP2_remeta_anno_file.txt {AOU_DIR}/AOU_remeta_anno_file.txt > {MERGE_DIR}/GP2_AOU_remeta_anno_file.txt
    

### Run meta-analysis

In [ ]:
# run remeta gene to get burden results
for ancestry in ["AAC", "AFR", "AJ", "AMR", "EUR"]:
    GP2_DIR="/home/jupyter/GP2_remeta_final"
    AOU_DIR=f"/home/jupyter/ITSN1_LD_Matrices/{ancestry}"
    MERGE_DIR="/home/jupyter/ITSN1_LD_Matrices/aou_gp2_merge_remeta_files"
    
    WORK_DIR=f"/home/jupyter/ITSN1_LD_Matrices/{ancestry}"
    
    ! ~/tools/remeta/build/remeta gene \
    --htp {MERGE_DIR}/{ancestry}_htp_GP2.txt {MERGE_DIR}/{ancestry}_htp_AOU.txt \
    --ld-prefix {GP2_DIR}/{ancestry}.GP2_WGS {AOU_DIR}/{ancestry}_AOU.cohort \
    --cohorts GP2_WGS AOU \
    --set-list {MERGE_DIR}/{ancestry}_set_list.txt \
    --extract {MERGE_DIR}/{ancestry}_extract.txt \
    --chr 21 \
    --anno-file {MERGE_DIR}/GP2_AOU_remeta_anno_file.txt \
    --mask-def /home/jupyter/ITSN1_LD_Matrices/mask_file.txt \
    --trait-name PHENO \
    --trait-type bt \
    --burden-aaf-bins 0.99 \
    --acatv-max-aaf 0.99 \
    --skip-acatv \
    --skato-max-aaf 0.01 \
    --out {MERGE_DIR}/{ancestry}_AOU_GP2

### Generate Output Files

In [ ]:
# merge htp files into one


import glob
import os
import pandas as pd

# read in annotation file
WORK_DIR = "/home/jupyter/ITSN1_LD_Matrices/aou_gp2_merge_remeta_files/"
anno = pd.read_csv(f"{WORK_DIR}/GP2_AOU_remeta_anno_file.txt", sep="\t", names=["Name", "Gene", "Annotation"])

# Find all CSV files in the current folder (change to '*.xlsx' for Excel)
for ancestry in ["AAC", "AFR", "AJ", "AMR", "EUR"]:
    all_files = glob.glob(f"{WORK_DIR}/{ancestry}_htp_*")
    
    li = []
    
    for filename in all_files:
        # Read the file
        df = pd.read_csv(filename, sep="\t")

        # merge with anno file for annotation, then restrict to LoF columns

        df = df.merge(anno, on="Name", how="left")

        
        # Create a new column with just the file's clean name
        df["Ancestry"] = ancestry
        
        li.append(df)
    
    # Concatenate all dataframes together
    combined_df = pd.concat(li, axis=0, ignore_index=True)
    
    # Save to a new file
    combined_df.to_csv(f"{WORK_DIR}/{ancestry}_GP2_AOU_htp_output.txt", sep="\t", index=False)


In [14]:
# concatenate output
! cat {WORK_DIR}/*_GP2_AOU_LoF_htp_output.txt > {WORK_DIR}/all_ancestry_GP2_AOU_lof_htp.txt